In [7]:
# Preparing pathing
%load_ext autoreload
%autoreload 2
from titanic_ml import paths
import matplotlib.pyplot as plt
import pandas as pd
from titanic_ml.common.data.eda import summarize_categorical_column, summarize_numerical_column
from titanic_ml.common.data.eda import run_eda 


from ast import For
from cmath import exp

from titanic_ml.common.experiments.runner import run_experiments, run_experiment_group_workflow
from titanic_ml.common.experiments.config import ALL_EXPERIMENTS
from titanic_ml.common.experiments.report import experiment_report, experiment_group_summary_report, baseline_summary_to_markdown, workflow_report
from titanic_ml.common.experiments.save_load import save_results, load_results, save_configs, load_configs
from titanic_ml.common.experiments.compare import leaderboard, compare_experiment_groups, summarize_group_comparison, titanic_notes_leaderboard


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
TARGET = "Survived"
ALL_EXPERIMENTS = ALL_EXPERIMENTS
for experiment_name, exp_config in ALL_EXPERIMENTS.items():
    print(f"Experiment: {experiment_name}")

train_df = pd.read_csv(paths.TRAIN_PATH)

exp_configs = ALL_EXPERIMENTS["fe02__Has_Cabin"]

# Line to rerun all experiments to update the results with the latest code changes. This will take a while.
# Uncomment to run all experiments and update results.
for Name, exp_config in ALL_EXPERIMENTS.items():
    print(f"Running {Name} experiments...")
    exp_result = run_experiments(train_df, exp_config, target=TARGET, verbose=True, debug=True)
    save_results(exp_result)
    save_configs(exp_config)



Experiment: baseline__raw
Experiment: fe01__family
Experiment: fe02__Has_Cabin
Experiment: fe03__Title
Running baseline__raw experiments...
running exp: {'name': 'baseline__raw__logreg', 'features': ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked'], 'feature_engineering': [], 'preprocessing': {'numeric_features': ['Age', 'SibSp', 'Parch', 'Fare'], 'onehot_features': ['Sex', 'Embarked'], 'ordinal_features': ['Pclass'], 'numeric_imputer': 'median', 'categorical_imputer': 'most_frequent', 'scaler': 'standard'}, 'model_name': 'logreg', 'model_params': {'max_iter': 1000, 'random_state': 42}, 'evaluation': {'method': 'cross_validation', 'cv': 5, 'scoring': ['accuracy', 'precision', 'recall', 'f1'], 'return_train_score': True, 'n_jobs': -1}, 'notes': 'Base logistic regression, using raw configuration. Baseline for comparison.', 'stage': 'baseline', 'feature_group': 'raw', 'group': 'baseline__raw'}
Running experiment: baseline__raw__logreg

Experiment 'baseline__raw__logreg' resul

In [9]:
print("Experiment Configurations:")
print(exp_configs)
for exp_config in exp_configs:
    print(exp_config)

Experiment Configurations:
[{'name': 'fe02__Has_Cabin__logreg', 'features': ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'Has_Cabin'], 'feature_engineering': [<function add_has_cabin at 0x000001FA225FC3A0>], 'preprocessing': {'numeric_features': ['Age', 'SibSp', 'Parch', 'Fare'], 'onehot_features': ['Sex', 'Embarked'], 'ordinal_features': ['Pclass', 'Has_Cabin'], 'numeric_imputer': 'median', 'categorical_imputer': 'most_frequent', 'scaler': 'standard'}, 'model_name': 'logreg', 'model_params': {'max_iter': 1000, 'random_state': 42}, 'evaluation': {'method': 'cross_validation', 'cv': 5, 'scoring': ['accuracy', 'precision', 'recall', 'f1'], 'return_train_score': True, 'n_jobs': -1}, 'notes': 'Feature engineering 02: adds Has_Cabin feature, which indicates whether the passenger had a known cabin or not. This is a simple binary feature that attempts to check if the missingess is a signal in itself.', 'stage': 'fe02', 'feature_group': 'Has_Cabin', 'group': 'fe02__Has_Cabin'

In [10]:
# Work flow for running an experiment group, comparing it to the baseline, and generating a report. 
# This is the main workflow for analyzing the results of an experiment group and generating insights from it.
workflow = run_experiment_group_workflow(
    df=train_df,
    experiment_configs=exp_configs,
    target=TARGET,
)

print("Workflow completed. Here are the results:")
print("Comparison between baseline and feature engineering group:")
print(workflow["comparison"])
print("Summary of comparison:")
print(workflow["summary"])
print("Leaderboard:")
print(workflow["leaderboard"])

running exp: {'name': 'fe02__Has_Cabin__logreg', 'features': ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'Has_Cabin'], 'feature_engineering': [<function add_has_cabin at 0x000001FA225FC3A0>], 'preprocessing': {'numeric_features': ['Age', 'SibSp', 'Parch', 'Fare'], 'onehot_features': ['Sex', 'Embarked'], 'ordinal_features': ['Pclass', 'Has_Cabin'], 'numeric_imputer': 'median', 'categorical_imputer': 'most_frequent', 'scaler': 'standard'}, 'model_name': 'logreg', 'model_params': {'max_iter': 1000, 'random_state': 42}, 'evaluation': {'method': 'cross_validation', 'cv': 5, 'scoring': ['accuracy', 'precision', 'recall', 'f1'], 'return_train_score': True, 'n_jobs': -1}, 'notes': 'Feature engineering 02: adds Has_Cabin feature, which indicates whether the passenger had a known cabin or not. This is a simple binary feature that attempts to check if the missingess is a signal in itself.', 'stage': 'fe02', 'feature_group': 'Has_Cabin', 'group': 'fe02__Has_Cabin'}
running exp: 

ValueError: Comparison group 'fe02__Has_Cabin' not found.

In [ ]:
# Generate a full report for the workflow, including the comparison, summary, and leaderboard. 
# This will be a markdown report that can be easily shared and visualized.
# Mainly used for generating the report for the notebook, but can also be used for generating reports for individual experiment groups or comparisons.
full_report = workflow_report(workflow)

print("Full workflow report:")
print()
print('Report')
print(full_report['report'])
print()
print('Leaderboard')
print(full_report['leaderboard'])

Full workflow report:

Report
### fe01__family

_Description pending._

<details>
<summary>Comparison details</summary>

#### Comparison vs baseline__raw

| reference_group   | compare_group   | model_name    |   test_accuracy_mean_reference |   test_accuracy_mean_compare |   test_accuracy_mean_delta |   test_f1_mean_reference |   test_f1_mean_compare |   test_f1_mean_delta |
|:------------------|:----------------|:--------------|-------------------------------:|-----------------------------:|---------------------------:|-------------------------:|-----------------------:|---------------------:|
| baseline__raw     | fe01__family    | logreg        |                          0.786 |                        0.795 |                      0.009 |                    0.713 |                  0.721 |                0.008 |
| baseline__raw     | fe01__family    | knn           |                          0.809 |                        0.805 |                     -0.004 |                    0.742

In [11]:
# Reminder of how to run a single experiment if needed. 
# This is useful for when we want to generate reports or comparisons without rerunning all experiments.

# exp_config = [exp_config]
exp_result = run_experiments(train_df, exp_config, target=TARGET,)
exp_report = experiment_report(exp_result, exp_config, print_report=True)
# save_results(exp_result)
# save_configs(exp_config)

running exp: name


TypeError: string indices must be integers

In [ ]:
# Summary for baseline experiment to use in report, 
# since we can't compare it to itself.

# baseline_summary = baseline_summary_to_markdown(exp_result)
# print("Baseline summary:")
# print(baseline_summary)

In [ ]:
# Reminder for how to load results and configs if needed. 
# This is useful for when we want to generate reports or comparisons without rerunning all experiments.

# all_results = load_results()
# print("Loaded results:")
# print(all_results)

# all_configs = load_configs()
# print("Loaded configs:")
# print(all_configs)

Loaded results:
                      experiment     stage feature_group     model_name  \
0           fe01__family__logreg      fe01        family         logreg   
1              fe01__family__knn      fe01        family            knn   
2              fe01__family__svc      fe01        family            svc   
3    fe01__family__decision_tree      fe01        family  decision_tree   
4    fe01__family__random_forest      fe01        family  random_forest   
5      fe01__family__extra_trees      fe01        family    extra_trees   
6              fe01__family__xgb      fe01        family            xgb   
7          baseline__raw__logreg  baseline           raw         logreg   
8             baseline__raw__knn  baseline           raw            knn   
9             baseline__raw__svc  baseline           raw            svc   
10  baseline__raw__decision_tree  baseline           raw  decision_tree   
11  baseline__raw__random_forest  baseline           raw  random_forest   
12    bas